# 01.04 · Structured output you can trust

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 01.01–01.03. Basic Pydantic helps but is taught here. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Define** output schemas with Pydantic v2 (descriptions, `Literal` enums, nullable fields, validators), `TypedDict`, and raw JSON Schema, and know what each gives you.
2. **Use** `with_structured_output` and **choose** a method: `function_calling`, `json_schema` (strict or best-effort) or `json_mode`, per provider and model.
3. **Find** each integration's default method in code, and spot where docs and code disagree.
4. **Handle** failures without crashing: `include_raw=True`, `parsing_error`, and a retry that feeds the error back to the model.
5. **Design** schemas that work in strict mode (all fields required, nullable instead of optional, closed objects).

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Models → Structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output) | **Main page.** Pydantic / TypedDict / JSON Schema, `method`, `include_raw` |
| [Structured output (agents)](https://docs.langchain.com/oss/python/langchain/structured-output) | `ToolStrategy` vs `ProviderStrategy` for `create_agent` (used in Module 02) |
| [Groq structured outputs](https://console.groq.com/docs/structured-outputs) | Strict vs best-effort mode, supported models, schema requirements |
| [`ChatGroq.with_structured_output` reference](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) | Methods and the `strict` flag |
| [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/) · [fields](https://docs.pydantic.dev/latest/concepts/fields/) · [validators](https://docs.pydantic.dev/latest/concepts/validators/) | The schema language you'll use most |

---
## Step 0 · Install and set up

`pydantic` comes with `langchain-core`, so there's nothing extra to install.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "pydantic",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Why structured output?

In Lesson 01.03 you parsed `"Bug."` into `bug` with string cleanup and an allow-list. That works for one label. For an object with ten fields (names, dates, lists, numbers), hand-written parsing becomes the most fragile part of the app.

**Structured output** turns the model's answer into a contract: you describe the shape with a schema; LangChain tells the provider about it, parses the reply, and (with Pydantic) **validates** it. Your code receives a Python object or a clear error, never "almost JSON".

Our running example: extract event details from messy invitation text.

In [ ]:
# 🧪 Offline: the input we'll extract from (synthetic).
INVITATION = """Hi team! Quick reminder: our Q4 planning workshop is on Thursday 2026-10-22,
starting 9:30 in the morning at the Medellín office (3rd floor, room Guayacán).
Ana, Luis and Mariana will present. Bring laptops. It'll also be streamed on Meet for remote folks."""
print(INVITATION)

---
## Part 2 · A Pydantic schema is a prompt *and* a validator

A Pydantic model does two jobs at once:

1. **Prompt:** LangChain converts it to JSON Schema and sends it to the provider. Class docstring, field names and `Field(description=...)` are **instructions the model reads**. Write them for the model.
2. **Validator:** the reply is parsed into the class. Wrong types or values raise `ValidationError` instead of slipping into your app.

In [ ]:
# 🧪 Offline: the schema. Descriptions are instructions for the model.
from typing import Literal

from pydantic import BaseModel, Field, field_validator


class EventDetails(BaseModel):
    """Details of ONE event extracted from an invitation. Use null when the text doesn't say."""

    title: str = Field(description="Short name of the event, as written")
    date: str = Field(description="Event date in ISO format YYYY-MM-DD")
    start_time: str | None = Field(description="24-hour HH:MM, or null if not stated")
    location: str | None = Field(description="Physical place, or null if online only / not stated")
    attendance: Literal["in_person", "online", "hybrid"] = Field(description="How people can attend")
    presenters: list[str] = Field(description="First names of people presenting; empty list if none")

    @field_validator("date")
    @classmethod
    def date_is_iso(cls, value: str) -> str:
        """Reject anything that isn't YYYY-MM-DD (runs in Python, after the model answers)."""
        from datetime import date

        date.fromisoformat(value)   # Raises ValueError for bad formats.
        return value

In [ ]:
# 🧪 Offline: validation in action. One good dict, one with three problems.
from pydantic import ValidationError

good = {"title": "Q4 planning workshop", "date": "2026-10-22", "start_time": "09:30",
        "location": "Medellín office, room Guayacán", "attendance": "hybrid", "presenters": ["Ana", "Luis"]}
bad = good | {"date": "Thursday 22", "attendance": "both", "presenters": "Ana, Luis"}

print(EventDetails.model_validate(good))
try:
    EventDetails.model_validate(bad)
except ValidationError as error:
    print(f"\n{error.error_count()} validation errors:")
    for item in error.errors():
        print(f"  {item['loc'][0]:<11} {item['msg']}")

### What just happened?

- The good dict became an `EventDetails` object with attribute access (`event.date`) and type hints your IDE understands.
- The bad dict produced **three** precise errors: the date validator, the `Literal` enum, and the list type. This is the safety net between the model and your code.
- The `field_validator` runs **in Python, after** the model replies. It is **not** sent to the provider; only types, enums and descriptions are. Keep that difference in mind for Part 6.

---
## Part 3 · What actually gets sent: JSON Schema

`with_structured_output` converts your class to JSON Schema. Look at it: this is what the model sees. In **strict** mode, providers require every property to be `required` and objects closed with `additionalProperties: false` ([Groq schema requirements](https://console.groq.com/docs/structured-outputs#schema-requirements)). LangChain applies that transformation when you pass `strict=True`.

In [ ]:
# 🧪 Offline: the JSON Schema LangChain derives (strict variant), the same function ChatGroq uses.
import json

from langchain_core.utils.function_calling import convert_to_json_schema

strict_schema = convert_to_json_schema(EventDetails, strict=True)
print("required            :", strict_schema["required"])
print("additionalProperties:", strict_schema["additionalProperties"])
print("start_time          :", json.dumps(strict_schema["properties"]["start_time"]))
print("attendance          :", json.dumps(strict_schema["properties"]["attendance"]))

### What just happened?

- **All six** fields are `required`, even `start_time` and `location`. Strict mode has no optional fields; instead, a field can be **required but nullable**: `anyOf: [string, null]`. The model must always answer, and `null` is a valid answer.
- That's why our schema writes `start_time: str | None = Field(...)` **without a default**. It means "you must tell me, and 'unknown' is allowed". It's better than letting the model silently omit fields, and much better than letting it **invent** a value to fill a required string.
- `attendance` became an `enum`: with constrained decoding, the model literally cannot produce another value.

---
## Part 4 · Three methods, and the defaults hiding in code

| `method=` | How it works | Guarantees | Groq support (verified 2026-10-11) |
|---|---|---|---|
| `"function_calling"` | Pretends your schema is a tool and forces the model to "call" it | Valid JSON args; schema adherence is best-effort | All tool-calling models |
| `"json_schema"` | The provider's native structured output (`response_format`) | With `strict=True`: **constrained decoding**, output always matches the schema | Strict: `gpt-oss-20b`, `gpt-oss-120b`. Best-effort: also `qwen3.8-27b`, `gpt-oss-safeguard-20b` |
| `"json_mode"` | Only guarantees *some* valid JSON object; you describe the schema in the prompt | Weakest | Most models |

Two Groq limits from the same page: **streaming and tool use are not supported together with Structured Outputs**.

Which method do you get if you don't choose? Ask the code.

In [ ]:
# 🧪 Offline: read each integration's DEFAULT method from its signature.
import inspect

from langchain_groq import ChatGroq

def default_method(cls) -> str:
    """Return the default value of `method` in cls.with_structured_output."""
    return inspect.signature(cls.with_structured_output).parameters["method"].default

print("ChatGroq              :", default_method(ChatGroq))
for module, name in (("langchain_openai", "ChatOpenAI"), ("langchain_google_genai", "ChatGoogleGenerativeAI")):
    try:
        cls = getattr(__import__(module), name)
        print(f"{name:<22}: {default_method(cls)}")
    except ImportError:
        print(f"{name:<22}: (package not installed: {module.replace('_', '-')})")

In [ ]:
# 🧪 Offline: which models does langchain-groq allow strict=True for? (A private constant: read, don't import in your app.)
import langchain_groq.chat_models as groq_chat_models

print(sorted(groq_chat_models._STRICT_STRUCTURED_OUTPUT_MODELS))

### What just happened? Three findings worth a research-log entry

1. **`ChatGroq` defaults to `function_calling`.** If you want Groq's constrained decoding you must ask: `method="json_schema", strict=True`.
2. **`ChatOpenAI` defaults to `json_schema` in code** (if you have `langchain-openai` installed you saw it), while the [ChatOpenAI integration page](https://docs.langchain.com/oss/python/integrations/chat/openai) says the method "defaults to using function calling" (checked 2026-10-11). The code wins. **Never rely on a default you haven't checked: pass `method=` explicitly.**
3. **Strict mode support disagrees across sources.** Groq's [supported-models table](https://console.groq.com/docs/structured-outputs#supported-models) lists `qwen/qwen3.8-27b` for strict mode; the comparison table further down the same page says "Limited (GPT-OSS 20B, 120B)"; and `langchain-groq` 1.1.3 only enables strict for the two `gpt-oss` models, **silently dropping `strict=True` for anything else** (see the `with_structured_output` source). For the course: strict mode on `gpt-oss` only, and validate everything with Pydantic anyway.

---
## Part 5 · Live comparison of the three methods

`include_raw=True` changes the return value into a dict: `{"raw": AIMessage, "parsed": object | None, "parsing_error": Exception | None}`. A parsing failure is **returned instead of raised**, which is what you want in batch jobs, and you keep `raw.usage_metadata` for cost.

In [ ]:
# 🧪 Offline: json_mode needs the schema in the prompt. Build that prompt once.
from langchain.messages import HumanMessage, SystemMessage

SYSTEM = "Extract the event details from the user's text. Use null when the text doesn't say."
JSON_MODE_SYSTEM = SYSTEM + "\nReply with ONE JSON object matching this JSON Schema:\n" + json.dumps(strict_schema)
messages = [SystemMessage(SYSTEM), HumanMessage(INVITATION)]
json_mode_messages = [SystemMessage(JSON_MODE_SYSTEM), HumanMessage(INVITATION)]
print(f"System prompt: {len(SYSTEM)} characters normally, {len(JSON_MODE_SYSTEM)} for json_mode (it carries the schema)")

In [ ]:
# 💸 Calls Groq 3 times (fast model), ~2.5k tokens total.
import pandas as pd
from IPython.display import display

from mastery.models import get_model

CONFIGS = [("function_calling", {}, messages), ("json_schema", {"strict": True}, messages),
           ("json_mode", {}, json_mode_messages)]
rows, outputs = [], {}
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = get_model("fast", temperature=0, reasoning_effort="low")
    for method, extra, msgs in CONFIGS:
        structured = llm.with_structured_output(EventDetails, method=method, include_raw=True, **extra)
        result = structured.invoke(msgs)
        outputs[method] = result
        usage = result["raw"].usage_metadata or {}
        rows.append({"method": method, "valid": result["parsing_error"] is None,
                     "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
                     "error": repr(result["parsing_error"])[:60] if result["parsing_error"] else ""})
    display(pd.DataFrame(rows))

In [ ]:
# 🧪 Look at what each method extracted (only runs if the previous cell did).
for method, result in outputs.items():
    parsed = result["parsed"]
    print(f"{method:<17}", parsed.model_dump() if parsed else f"❌ {type(result['parsing_error']).__name__}")

### What just happened?

- With a key, you typically get **three valid objects**: date `2026-10-22`, start time `09:30` (converted from "9:30 in the morning"), attendance `hybrid` (office + Meet), presenters Ana, Luis, Mariana.
- **`json_mode` costs the most input tokens** because the schema travels inside the prompt, and it's the only one with no schema enforcement at all.
- `function_calling` and `json_schema` send the schema in dedicated API fields. Strict `json_schema` adds the guarantee that the output matches the schema.
- Valid ≠ correct. All three can produce a perfectly valid object with a wrong value (e.g. `attendance: in_person`). Schema adherence is not accuracy; Project 01 measures both.

---
## Part 6 · When parsing fails: catch, feed back, retry

Failures still happen: best-effort modes can break the schema, a Python validator (like `date_is_iso`) can reject a value the schema allowed, or the model refuses. The robust pattern:

1. Call with `include_raw=True`, so failures are values, not exceptions.
2. If `parsing_error` is set, append the error to the conversation and ask again **once or twice**.
3. Still failing? Return `None` and **count it**. Don't crash the batch, don't hide it.

To test this logic offline we use a **fake structured model**: a `RunnableLambda` that returns the same dict shape, failing the first time.

In [ ]:
# 🧪 Offline: the retry helper. Works with any runnable that returns the include_raw dict.
from langchain.messages import AIMessage


def extract_with_retry(structured_model, messages: list, max_attempts: int = 2):
    """Invoke a structured model (include_raw=True); on a parsing error, show it to the model and retry.

    Returns:
        (parsed object or None, number of attempts used)
    """
    conversation = list(messages)
    for attempt in range(1, max_attempts + 1):
        result = structured_model.invoke(conversation)
        if result["parsing_error"] is None:
            return result["parsed"], attempt
        # Feed the exact error back. Short and specific beats "try again".
        conversation += [AIMessage(result["raw"].text or "(no text)"),
                         HumanMessage(f"Your answer was invalid: {result['parsing_error']}. Return corrected output.")]
    return None, max_attempts

In [ ]:
# 🧪 Offline: a fake structured model that fails once (bad date), then succeeds.
from langchain_core.runnables import RunnableLambda

calls = {"n": 0}

def fake_structured(conversation: list) -> dict:
    calls["n"] += 1
    candidate = good | ({"date": "Thursday 22"} if calls["n"] == 1 else {})
    try:
        return {"raw": AIMessage(json.dumps(candidate)), "parsed": EventDetails.model_validate(candidate), "parsing_error": None}
    except ValidationError as error:
        return {"raw": AIMessage(json.dumps(candidate)), "parsed": None, "parsing_error": error}

parsed, attempts = extract_with_retry(RunnableLambda(fake_structured), messages)
print(f"attempts={attempts}  parsed date={parsed.date if parsed else None}")

### What just happened?

The first answer failed the `date_is_iso` validator; the helper sent the error back and the second answer passed. `attempts=2`. With a real model, the feedback message usually fixes format problems in one retry.

> **In agents, you won't write this loop.** `create_agent(response_format=ToolStrategy(Schema))` retries structured-output errors for you with a `handle_errors` option, and `ProviderStrategy` uses native structured output when the model supports it ([Structured output for agents](https://docs.langchain.com/oss/python/langchain/structured-output)). Module 02 uses them. Understanding the manual version tells you what those options do.

---
## Part 7 · `TypedDict` and raw JSON Schema

| Schema type | You get back | Validation | Use when |
|---|---|---|---|
| Pydantic `BaseModel` | An instance of your class | ✅ types, enums, validators | Default choice in Python |
| `TypedDict` | A plain `dict` | ❌ none (type hints only) | You need dicts and validate elsewhere |
| JSON Schema `dict` | A plain `dict` | ❌ none | The schema comes from outside (an API spec, another language) |

In [ ]:
# 🧪 Offline: the same schema as a TypedDict; descriptions go in Annotated[..., ..., "description"].
from typing import Annotated

from typing_extensions import TypedDict


class EventDict(TypedDict):
    """Details of ONE event extracted from an invitation."""

    title: Annotated[str, ..., "Short name of the event"]
    date: Annotated[str, ..., "ISO date YYYY-MM-DD"]
    attendance: Annotated[Literal["in_person", "online", "hybrid"], ..., "How people can attend"]


print(json.dumps(convert_to_json_schema(EventDict), indent=1)[:400])

The `...` in `Annotated[str, ..., "description"]` marks the field as required (the [Models page](https://docs.langchain.com/oss/python/langchain/models#structured-output) uses this form). Calling `llm.with_structured_output(EventDict)` would return a **dict**, with no validation. If the model returns `"date": "Thursday"`, nothing complains. That's why the course defaults to Pydantic.

---
## Part 8 · Schema design checklist

1. **Name and describe fields for the model.** `date` + "ISO format YYYY-MM-DD" beats `d`.
2. **Use `Literal` enums** for categories, and include an escape value (`"unknown"`, `"other"`) so the model isn't forced to guess.
3. **Required + nullable** (`X | None` without a default) for facts that may be missing. It works in strict mode and makes "not stated" explicit.
4. **Keep it flat-ish.** Nesting is fine (Project 01 nests a salary range), but deep trees cost tokens and accuracy.
5. **Put units and formats in names or descriptions:** `salary_min` + `currency` + `period`, not `salary: "about 50k"`.
6. **Validate what the schema can't express** with Pydantic validators, and handle the error (Part 6).
7. **Pass `method=` explicitly**, and `strict=True` where supported.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| 400 error: schema invalid in strict mode | Optional field with a default, or an object without `additionalProperties: false` | Required + nullable fields; let LangChain build the schema with `strict=True` |
| `strict=True` seems to have no effect | Model not in `langchain-groq`'s strict list: the flag is dropped silently | Use `gpt-oss-20b`/`120b` for strict; validate anyway |
| `OutputParserException` / `ValidationError` crashes a loop | `include_raw=False` raises on failure | `include_raw=True`, count failures |
| Output is valid JSON but missing your fields | `json_mode` without the schema in the prompt | Put the schema in the system prompt, or use `json_schema` |
| Streaming with structured output doesn't work on Groq | Not supported with Structured Outputs | Don't stream structured calls; stream a plain-text summary instead (Project 01) |
| Dict instead of an object | You passed a `TypedDict` or JSON Schema | Use a Pydantic model for validation |

## ⚠️ Production notes

> ⚠️ **Production note: valid is not correct.**
> This lesson checks that outputs *parse*. In production: also measure field-level accuracy on a labelled set (Project 01, Module 12), monitor the parse-failure rate per model and prompt version, and route low-confidence or invalid results to a human or a fallback model instead of retrying forever.

> ⚠️ **Production note: depending on a private constant.**
> Part 4 reads `langchain_groq.chat_models._STRICT_STRUCTURED_OUTPUT_MODELS` to learn something. The leading underscore means "private, may change without notice". Never import it in application code; re-check the behavior when you upgrade.

## 🏋️ Exercises

1. **(Easy)** Add a field `organizer_email: str | None` to `EventDetails`. Re-run Parts 3 and 5. Does the model return `null` (the invitation has no email) or invent one?
2. **(Easy)** Run Part 5 with `get_model("reasoning")`. Compare tokens and results.
3. **(Medium)** Write a second invitation that's **online only** and has **no start time**. Check that `location` and `start_time` come back as `None`, not invented values. If not, improve the descriptions.
4. **(Stretch)** Extract a **list** of events: define `class EventList(BaseModel): events: list[EventDetails]` and feed a text with two events. Then try `qwen/qwen3.8-27b` (`get_model("vision")`, best-effort mode only) with `method="json_schema"` and compare validity.

<details>
<summary><b>Show solutions</b></summary>

**3.**
```python
ONLINE = "Lunch & learn on prompt testing, 2026-11-05, fully online on Meet. Speaker: Sofía."
if HAVE_GROQ_KEY:
    structured = get_model("fast", temperature=0, reasoning_effort="low").with_structured_output(
        EventDetails, method="json_schema", strict=True)
    print(structured.invoke([SystemMessage(SYSTEM), HumanMessage(ONLINE)]))
```
Expect `start_time=None`, `location=None`, `attendance='online'`, `presenters=['Sofía']`. If the model invents "12:00" ("lunch" suggests noon!), strengthen the description: *"null unless a time is written explicitly"*.

**4.**
```python
class EventList(BaseModel):
    """All events mentioned in the text, in order."""
    events: list[EventDetails]

TWO = INVITATION + "\nAlso: retro on 2026-10-30 at 16:00, online only."
if HAVE_GROQ_KEY:
    result = get_model("fast", temperature=0).with_structured_output(
        EventList, method="json_schema", strict=True, include_raw=True).invoke(TWO)
    print(result["parsed"])
```
</details>

## ✅ Checkpoint

- [ ] I can explain why a Pydantic schema is both a prompt and a validator, and what is (and isn't) sent to the provider.
- [ ] I can write a strict-mode-compatible schema (required + nullable, enums, closed objects).
- [ ] I know `ChatGroq` defaults to `function_calling` and always pass `method=` explicitly.
- [ ] I can get failures as values with `include_raw=True` and retry with error feedback.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Which Groq models support **strict** `json_schema` today? Compare three sources: the [Groq structured outputs](https://console.groq.com/docs/structured-outputs) page, the model's [model card](https://console.groq.com/docs/models), and what `langchain-groq` actually sends (`include_raw=True` + the mock or a real call). Log any disagreement in [`docs/research-log.md`](../../docs/research-log.md); disagreements between docs and code are exactly what Lesson 00.05 trained you to find.

## Summary

- Structured output turns answers into validated objects. Pydantic is the default; `TypedDict` and JSON Schema return unvalidated dicts.
- `json_schema` + `strict=True` (on supported models) gives constrained decoding; `function_calling` is the portable default for `ChatGroq`; `json_mode` is the weakest.
- Docs and code disagree about defaults and supported models: check the code, pass options explicitly.
- Valid ≠ correct: measure accuracy separately. Count failures; never let one crash a batch.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.05 · Streaming, batching, and async basics** → [`01-05-streaming-and-batching.ipynb`](01-05-streaming-and-batching.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)